# Module 13: Baseline Forecasts You Must Beat

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Before fitting anything, work out what a method with no ideas in it would
predict. That number is the bar. A model that does not clear it is not a model,
it is a liability, because someone will maintain it for years.

**About 15 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
from pathlib import Path

import numpy as np
import pandas as pd

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

print("reading from:", BASE)

monthly = pd.read_csv(BASE + "agency_monthly.csv")
final = monthly[monthly["provisional"] == 0]        # never fit on unfinished months


def series(agency_id, column="n_uof"):
    """One agency's monthly series, indexed by date."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    return pd.Series(d[column].values, dtype=float,
                     index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())


def rate(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    return pd.Series((100 * d["n_uof"] / d["n_arrests"]).values,
                     index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())


grandview = series("A012")
print(f"{len(grandview)} months, {grandview.index.min():%Y-%m} to {grandview.index.max():%Y-%m}")


def split(s, end="2024-12", horizon=12):
    """Train on everything up to `end`, test on the next `horizon` months."""
    train = s.loc[:end]
    test = s.loc[pd.Timestamp(end) + pd.offsets.MonthBegin(1):][:horizon]
    return train, test


def mae(actual, pred):
    return float(np.mean(np.abs(np.asarray(actual, float) - np.asarray(pred, float))))

## 2. Split by time, never at random

A random split lets the model learn from the future. Forecasting is judged on
whether you could have said this **in advance**, so the test period must come
after everything the model saw.

In [ ]:
grandview = series("A012")
train, test = split(grandview, end="2024-12")

print(f"train: {len(train)} months, {train.index[0]:%Y-%m} to {train.index[-1]:%Y-%m}")
print(f"test : {len(test)} months, {test.index[0]:%Y-%m} to {test.index[-1]:%Y-%m}")

## 3. Six baselines, none of them clever

Each is one line of arithmetic.

In [ ]:
drift = (train.iloc[-1] - train.iloc[0]) / (len(train) - 1)

baselines = {
    "last value": np.repeat(train.iloc[-1], 12),
    "mean of all history": np.repeat(train.mean(), 12),
    "rolling twelve month mean": np.repeat(train.iloc[-12:].mean(), 12),
    "same month last year": train.iloc[-12:].values,
    "drift": train.iloc[-1] + drift * np.arange(1, 13),
    "seasonal naive plus drift": train.iloc[-12:].values + drift * 12,
}

results = pd.Series({k: mae(test.values, v) for k, v in baselines.items()})
results.sort_values().round(2).to_frame("average error, incidents a month")

## 4. What the ranking says

**Seasonal naive plus drift wins**, and plain seasonal naive is barely behind
it. Both say the same thing: *whatever happened this month last year, expect
roughly that again*. For a series with a strong annual pattern that is a
genuinely good forecast, and it costs nothing to compute or explain.

**The mean of all history is the worst.** It ignores both the season and the
trend, so it is wrong in a different direction every month.

**Last value is poor too**, because December is a quiet month and repeating it
across a whole year predicts a permanent winter.

In [ ]:
comparison = pd.DataFrame({
    "what happened": test.values.astype(int),
    "same month last year": baselines["same month last year"].astype(int),
    "last value": baselines["last value"].astype(int),
}, index=[f"{d:%b}" for d in test.index])
comparison["error, seasonal naive"] = (comparison["what happened"]
                                       - comparison["same month last year"])
comparison

## 5. Why the baseline is the right bar, not the mean

It is tempting to judge a forecast by asking whether it is close. Close to what?
A forecast of a seasonal series can look impressive simply by reproducing the
season, which the baseline already does for free.

So the only question worth asking about a new method is: **does it beat the
baseline, on data it has not seen, by enough to be worth maintaining?**

In [ ]:
best_baseline = results.min()
for target in [0.9, 0.75, 0.5]:
    print(f"to cut the baseline error by {100 * (1 - target):.0f} percent, "
          f"a model needs an average error below {best_baseline * target:.1f}")

## 6. Record the baseline before you start

Write it down before fitting anything. It is remarkably easy to build a model,
see an error of 12, and feel satisfied without ever checking that the free
answer was 15.6.

In [ ]:
def baseline_table(s, end="2024-12", horizon=12):
    """Every baseline, scored, for any series."""
    train, test = split(s, end, horizon)
    if len(test) < horizon or len(train) < 24:
        return None
    d = (train.iloc[-1] - train.iloc[0]) / (len(train) - 1)
    cand = {
        "last value": np.repeat(train.iloc[-1], horizon),
        "mean of all history": np.repeat(train.mean(), horizon),
        "rolling twelve month mean": np.repeat(train.iloc[-12:].mean(), horizon),
        "same month last year": train.iloc[-12:].values[:horizon],
        "drift": train.iloc[-1] + d * np.arange(1, horizon + 1),
        "seasonal naive plus drift": train.iloc[-12:].values[:horizon] + d * 12,
    }
    return pd.Series({k: mae(test.values, v) for k, v in cand.items()}).sort_values()

In [ ]:
for aid in ["A012", "A002", "A008"]:
    r = baseline_table(series(aid))
    name = final[final["agency_id"] == aid]["agency_name"].iloc[0]
    print(f"{name:34s} best baseline: {r.index[0]:26s} error {r.iloc[0]:.2f}")

## Exercise

Run the baseline table on Orrindale, the eight officer department. Which baseline
wins, and what does the winner tell you?

In [ ]:
# Fill in the blank, then run.
AGENCY = None              # try "A006"

if AGENCY:
    r = baseline_table(series(AGENCY))
    print(r.round(3).to_string())
    print(f"\nmonthly average over the training period: "
          f"{split(series(AGENCY))[0].mean():.2f}")
else:
    print("Set AGENCY above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
AGENCY = "A006"
```

A **flat** baseline wins, and the seasonal ones come last:

| baseline | average error |
|---|---|
| last value | 0.58 |
| drift | 0.58 |
| mean of all history | 0.62 |
| rolling twelve month mean | 0.69 |
| **same month last year** | **0.92** |

Every one of them is within half an incident of every other, on a series
averaging 0.81 incidents a month. Note that the ordering is the **reverse** of
Ashfell's: there, using the same month last year was the best available
answer; here it is the worst, because Orrindale has no seasonal shape large
enough to see through the noise, so reaching back twelve months just imports an
extra month of randomness.

That is the answer, and it is not a disappointing one. **Knowing that a series
is unforecastable is a result.** It saves building something that would be
maintained for years while adding nothing, and it redirects the question to a
time unit where an answer exists, as in
[Module 8](Module_08_Rolling_Statistics_And_Control_Limits.ipynb).

</details>

---

**Next:** [Module 14, Exponential Smoothing in Plain Language](Module_14_Exponential_Smoothing.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*